<a href="https://colab.research.google.com/github/PreethiR091/Deep-Learning/blob/main/Preethi_R_24BAD091_DL_Ex_no_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Preethi R 24BAD091

EXPERIMENT NO. 9 – TRANSFORMER-BASED LLM CHATBOT USING GRADIO

PART A: LOADING AND CONFIGURING A PRE-TRAINED LLM

Objective: Install the required libraries, load a pre-trained Transformer model and tokenizer, tokenize sample prompts and generate responses.

In [1]:
# Student Details
student_name = "Preethi R"
roll_no = "24BAD091"

# Install required libraries
!pip install -q transformers accelerate gradio

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# Select pre-trained model
model_name = "microsoft/DialoGPT-small"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Sample user prompt
prompt = "Hello, how are you?"

# Tokenize input
input_ids = tokenizer.encode(
    prompt + tokenizer.eos_token,
    return_tensors="pt"
).to(device)

print("Student Name:", student_name)
print("Roll Number:", roll_no)
print("Model Name:", model_name)
print("Device:", device)
print("Tokenized Input:", input_ids)

# Generate response
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True,
        top_p=0.92,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode generated response
response = tokenizer.decode(
    output[:, input_ids.shape[-1]:][0],
    skip_special_tokens=True
)

print("\nUser:", prompt)
print("Chatbot:", response)

config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  351MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Student Name: Preethi R
Roll Number: 24BAD091
Model Name: microsoft/DialoGPT-small
Device: cpu
Tokenized Input: tensor([[15496,    11,   703,   389,   345,    30, 50256]])


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



User: Hello, how are you?
Chatbot: I'm famous !


PART B: IMPLEMENTING CHATBOT FUNCTIONALITY

Objective: Create a chatbot function that accepts user queries, generates responses using the pre-trained LLM and maintains conversation history.

In [2]:
# Chatbot response function
def chatbot_response(user_query, history=None):

    if history is None:
        history = []

    # Prepare conversation history
    chat_input = torch.empty((1, 0), dtype=torch.long, device=device)

    for message in history:
        if isinstance(message, dict):
            content = message["content"]
        else:
            content = message[0]

        if content:
            tokens = tokenizer.encode(
                str(content) + tokenizer.eos_token,
                return_tensors="pt"
            ).to(device)

            chat_input = torch.cat((chat_input, tokens), dim=-1)

    # Tokenize current user query
    new_input = tokenizer.encode(
        user_query + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    chat_input = torch.cat((chat_input, new_input), dim=-1)

    # Limit conversation length
    chat_input = chat_input[:, -500:]

    # Generate chatbot response
    with torch.no_grad():
        output = model.generate(
            chat_input,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True,
            top_p=0.92,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode response
    response = tokenizer.decode(
        output[:, chat_input.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response


# Test chatbot
print("Student Name: Preethi R")
print("Roll Number: 24BAD091")

queries = [
    "Hello!",
    "What is artificial intelligence?",
    "What is deep learning?"
]

history = []

for query in queries:
    answer = chatbot_response(query, history)

    print("\nUser:", query)
    print("Chatbot:", answer)

    history.extend([
        {"role": "user", "content": query},
        {"role": "assistant", "content": answer}
    ])

Student Name: Preethi R
Roll Number: 24BAD091

User: Hello!
Chatbot: hello there ! :D DDDd I'm a girl . lt 3 Edit spelling of your name , sorry about that ... it's Aika XD and my IGN is Nami so no one notices xP edit names changed to japanese .. lolz ' Sighs again Tfw people can't read Pinguinetation Cxu hahahaha oopsie cx

User: What is artificial intelligence?
Chatbot: it seems we are in the same boat here as well haha ikr ? cxcxdkqcjswksvkjklfapkrjjknokkk kwaiii xxlxxxllp wootttoooooooooooodddooooxxxxxyyyooooooooooooooooooooo baaaaaa nuuuhhhhhzzzxeaxaheeeaaaasuaay yoloiii

User: What is deep learning?
Chatbot: Ohmccchasee tawniare u havellyfaklyoveeeee i seewwwinintanganepatwwithssiedbongadenetaoowamaya buihgate zacomeame gtgisecosomakebeating mochohypeopigarff 1carecraft eauightaloreunite okae fazalone


PART C: DEVELOPING AN INTERACTIVE INTERFACE

Objective: Develop an interactive chatbot interface using Gradio and connect it to the response-generation function.

In [4]:
import gradio as gr

# Create Gradio chatbot interface
chatbot = gr.ChatInterface(
    fn=chatbot_response,
    title="Transformer-Based LLM Chatbot",
    description="Student Name: Preethi R | Roll No: 24BAD091",
    textbox=gr.Textbox(
        placeholder="Enter your message here...",
        label="Your Message"
    )
)

# Launch chatbot
chatbot.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://23f4b4ff44cbc765f7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


PART D: APPLICATION TESTING AND DEPLOYMENT

Objective: Test the chatbot, observe its limitations, prepare project files and upload the application to GitHub or Hugging Face Spaces.

In [5]:
# Test chatbot with different inputs
test_queries = [
    "Hello!",
    "What is machine learning?",
    "Explain artificial intelligence."
]

print("Student Name: Preethi R")
print("Roll Number: 24BAD091")

for query in test_queries:
    response = chatbot_response(query, [])
    print("\nUser:", query)
    print("Chatbot:", response)

# Create app.py for deployment
app_code = '''
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def chatbot_response(user_query, history):

    chat_input = torch.empty((1, 0), dtype=torch.long, device=device)

    for message in history:
        content = message["content"]

        tokens = tokenizer.encode(
            str(content) + tokenizer.eos_token,
            return_tensors="pt"
        ).to(device)

        chat_input = torch.cat((chat_input, tokens), dim=-1)

    new_input = tokenizer.encode(
        user_query + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    chat_input = torch.cat((chat_input, new_input), dim=-1)
    chat_input = chat_input[:, -500:]

    with torch.no_grad():
        output = model.generate(
            chat_input,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True,
            top_p=0.92,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        output[:, chat_input.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response

chatbot = gr.ChatInterface(
    fn=chatbot_response,
    title="Transformer-Based LLM Chatbot"
)

chatbot.launch()
'''

with open("app.py", "w") as f:
    f.write(app_code)

# Create requirements.txt
requirements = """torch
transformers
gradio
accelerate
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("\nProject files created successfully.")
print("1. app.py")
print("2. requirements.txt")

Student Name: Preethi R
Roll Number: 24BAD091

User: Hello!
Chatbot: hello guys ! :D i am a girl and this is awesome . xoxo lt 3 edit also , hello fellow ladies aswell D: hope you dont mind me hijacking the post to make it clearer XD ... ? lolzololo oops typo thx btw kudos for noticing that one .. haha good job on your stream too hahahaha thanks again p3n

User: What is machine learning?
Chatbot: Not to mention it's the computer science course , so you'll likely need a good amount of experience with computers and algorithms . It might be useful for someone like me though ... :D ! ? haha D: xDDd I'm kidding not lol .. XD P3P0E4W8Y9A1QF55CX7UCC6HN5LZ95O

User: Explain artificial intelligence.
Chatbot: It's a joke about how the internet is just an endless source of information . This has been proven to be true for at least as long and it still baffles me sometimes ... s EDIT : grammar correction ? ! Edit edit .. sarcasm alert lol jk , sorry guysz i get jokes rn lt 3 xDeXSpAiRrEcTiBySt

Pr